# Exoplanet Detection from Light Curves

**Challenge 7 — Bharatiya Antariksh Hackathon 2026**

Pipeline: Data → Detrend → Period Search → Phase Fold → Feature Extraction → ML Classification → Parameter Estimation

In [5]:
# Install missing runtime dependency required by project modules
# (magic must appear at the top of the Jupyter cell)
%pip install astropy --quiet

import sys
import os
from pathlib import Path

# Resolve repository root by searching up from __file__ (if available) or cwd
# We pick the first ancestor that contains a 'src' directory (or common project markers).
try:
	base_path = Path(__file__).resolve()
except NameError:
	base_path = Path.cwd().resolve()

parent_dir = None
for p in [base_path] + list(base_path.parents):
	if (p / "src").exists() or (p / ".git").exists() or (p / "pyproject.toml").exists() or (p / "setup.py").exists():
		parent_dir = p
		break

# Fallback to base_path if nothing matched
if parent_dir is None:
	parent_dir = base_path

# Ensure the repository root is on sys.path so imports like `from src...` work
sys.path.insert(0, str(parent_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_utils import generate_synthetic_light_curve, fetch_kepler_data
from src.preprocessing import detrend_light_curve, find_best_period, phase_fold, extract_transit_shape
from src.features import extract_features_from_lc, build_feature_matrix
from src.model import train_classifier, predict_exoplanet, estimate_transit_parameters
from src.pipeline import generate_training_data

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'xgboost'

## 1. Generate / Fetch a Light Curve

In [ ]:
lc = generate_synthetic_light_curve(
    transit_depth=0.01,
    transit_duration=0.1,
    orbital_period=4.5,
    noise_scale=0.002,
)
lc.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(lc['time'], lc['flux'], 'b.', markersize=1, alpha=0.5)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Normalized Flux')
ax.set_title('Raw Light Curve')
ax.axhline(1.0, color='gray', ls='--', alpha=0.4)
plt.show()

## 2. Detrending

In [ ]:
flux_detrended = detrend_light_curve(lc['time'].values, lc['flux'].values)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax1.plot(lc['time'], lc['flux'], 'b.', markersize=1, alpha=0.5)
ax1.set_ylabel('Raw Flux')
ax1.axhline(1.0, color='gray', ls='--', alpha=0.4)

ax2.plot(lc['time'], flux_detrended, 'b.', markersize=1, alpha=0.5)
ax2.set_xlabel('Time (days)')
ax2.set_ylabel('Detrended Flux')
ax2.axhline(1.0, color='gray', ls='--', alpha=0.4)
plt.show()

## 3. Period Search (Lomb-Scargle)

In [ ]:
period_result = find_best_period(lc['time'].values, flux_detrended)
print(f"Best period: {period_result['period']:.4f} days")
print(f"Peak power: {period_result['power']:.4f}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(1/period_result['all_frequencies'], period_result['all_powers'], 'b-', alpha=0.6)
ax.set_xlabel('Period (days)')
ax.set_ylabel('Power')
ax.set_title('Lomb-Scargle Periodogram')
plt.show()

## 4. Phase Folding

In [ ]:
phase_sorted, flux_sorted, binned_flux = phase_fold(
    lc['time'].values, flux_detrended, period_result['period']
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(phase_sorted, flux_sorted, 'b.', markersize=2, alpha=0.3)
ax.plot(np.linspace(0, 1, len(binned_flux)), binned_flux, 'r-', linewidth=2)
ax.set_xlabel('Phase')
ax.set_ylabel('Normalized Flux')
ax.set_title(f'Phase Folded Light Curve (P = {period_result["period"]:.2f} d)')
ax.axhline(1.0, color='gray', ls='--', alpha=0.4)
plt.show()

## 5. Feature Extraction & Transit Shape

In [ ]:
features = extract_features_from_lc(lc['time'].values, lc['flux'].values)
pd.DataFrame([features]).T

## 6. Train Classifier

In [ ]:
lcs, labels = generate_training_data(num_realistic=80, num_noise=80)
feature_df = build_feature_matrix(lcs)

idx_map = {}
for i in range(80):
    idx_map[f'planet_{i:03d}'] = i
    idx_map[f'noise_{i:03d}'] = 80 + i
aligned_idx = [idx_map[lid] for lid in feature_df['id'] if lid in idx_map]
aligned_labels = labels.iloc[aligned_idx]

result = train_classifier(feature_df, aligned_labels, model_type='rf')

print(f"ROC-AUC: {result['roc_auc']:.4f}")
print(f"CV AUC: {result['cv_auc_mean']:.4f} +/- {result['cv_auc_std']:.4f}")
print(result['test_report'])

importances = pd.Series(result['feature_importances']).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
importances.head(15).plot.barh(ax=ax)
ax.set_xlabel('Importance')
ax.set_title('Top 15 Features')
plt.tight_layout()
plt.show()

## 7. Test Predictions

In [ ]:
test_lc = generate_synthetic_light_curve(
    transit_depth=0.015, transit_duration=0.08,
    orbital_period=3.2, noise_scale=0.002, random_seed=123
)

test_features = extract_features_from_lc(test_lc['time'].values, test_lc['flux'].values)
pred = predict_exoplanet(test_features, result['classifier'], result['scaler'], result['feature_cols'])
print(f"Prediction: {pred['label']} (confidence: {pred['confidence']:.2%})")

In [ ]:
params = estimate_transit_parameters(
    test_lc['time'].values, test_lc['flux'].values,
    period=test_features['best_period']
)
for k, v in params.items():
    print(f"  {k}: {v}")

## 8. Try with Real Kepler Data

Uncomment and run to fetch real Kepler data:

```python
kepler_lc = fetch_kepler_data('Kepler-10', quarter=1)
if kepler_lc is not None:
    time, flux = kepler_lc['time'].values, kepler_lc['flux'].values
    # mask nans
    mask = np.isfinite(time) & np.isfinite(flux)
    time, flux = time[mask], flux[mask]
    
    feats = extract_features_from_lc(time, flux)
    pred = predict_exoplanet(feats, result['classifier'], result['scaler'], result['feature_cols'])
    print(pred)
```

Known exoplanet hosts to try: `Kepler-10`, `Kepler-22`, `Kepler-186`, `Kepler-452`, `K2-3`, `TRAPPIST-1`